# 01 · Data Generation and Cleaning

# Project: Integrating Moral Values in Turkish EFL Classrooms

These notebooks are a reproducible analysis companion for the supplied project materials. They do not invent participant-level records. Appendix E/F provide aggregate frequencies with N=20; separate questionnaire notes contain percentages without an explicit denominator and conflict with the appendices on some items. Treat each source stream separately. The main manuscript describes a larger mixed-methods sample (the supplied abstract is truncated), so these appendix counts must not be silently generalized to the manuscript sample.

Run from any working directory. Generated files go to `./analysis_outputs` relative to the current working directory. Python 3.9+; dependencies: pandas, numpy, matplotlib (Notebook 3 only).

## Purpose and provenance
Create tidy, analysis-ready tables from the aggregate summaries transcribed in Appendix E and F of `Appendices_and_Questionnaires.docx`. Values are aggregate counts, **not reconstructed individual responses**. A separate supplementary-notes table preserves distinct, potentially conflicting percentage claims from `Questionaire.docx` / `Questionaire-(2).docx`.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
OUT = Path('analysis_outputs'); OUT.mkdir(exist_ok=True)

## Transcribed appendix summaries
Teacher and student frequency rows below are exactly the summary categories and counts reported in Appendices E/F. The source reports N=20 for teachers; the student denominator is not explicitly stated in the excerpt, so N=20 is used only where the table frequencies sum to 20 and is flagged as an inferred table total.

In [ ]:
teacher_rows = [
 ('Q1 responsibility','Fully / considerably responsible',14),
 ('Q2 essential value','Global citizenship',14),('Q2 essential value','Empathy',3),('Q2 essential value','Respect',2),('Q2 essential value','Honesty',1),
 ('Q3 curriculum guidance','No (insufficient guidance)',16),('Q3 curriculum guidance','Yes (sufficient guidance)',4),
 ('Q5 effective technique','Debates',9),('Q5 effective technique','Storytelling',6),('Q5 effective technique','Role-playing',3),('Q5 effective technique','Group work',2),
 ('Q7 main obstacle','Student resistance',10),('Q7 main obstacle','Time constraints',6),('Q7 main obstacle','Lack of training',4),
]
student_rows = [
 ('Story recall','Could recall a story with a moral message',17),('Story recall','Could not recall a specific story',3),
 ('Most-enjoyed story','The Boy Who Cried Wolf',6),('Most-enjoyed story','The Honest Woodcutter',5),('Most-enjoyed story','The Golden Eggs',4),('Most-enjoyed story','The Bear and Two Friends',3),('Most-enjoyed story','Other / none',2),
 ('Cultural perspective','English helps understand other values',15),('Cultural perspective','Not sure',4),('Cultural perspective','No',1),
 ('Story preference','Stories with a clear moral lesson',12),('Story preference','Student decides what is right',5),('Story preference','Both',3),
]
teachers = pd.DataFrame(teacher_rows, columns=['item','response','frequency'])
teachers['denominator'] = 20
teachers['percent'] = teachers.frequency / teachers.denominator * 100
students = pd.DataFrame(student_rows, columns=['item','response','frequency'])
students['denominator'] = students.groupby('item').frequency.transform('sum')
students['denominator_status'] = 'inferred from Appendix F frequency total; verify against source records'
students['percent'] = students.frequency / students.denominator * 100
supplementary = pd.DataFrame([
 ('Teacher relationship to moral education','70% reportedly said EFL should be separate from moral class',70,None,'Questionaire.docx and Questionaire-(2).docx; denominator not stated; conflicts with Appendix E Q1 framing'),
 ('Teacher material selection','Videos 50%; news 30%; new articles 20%',None,None,'Questionaire.docx and Questionaire-(2).docx; wording/categories ambiguous; denominator not stated'),
 ('Student ethical discourse frequency','40% reportedly answered “Yes most of the times”',40,None,'Questionaire.docx and Questionaire-(2).docx; denominator not stated'),
],columns=['item','reported_claim','percent','denominator','provenance_note'])
print('Teacher rows:',len(teachers),' N:',teachers.denominator.unique().tolist())
print(teachers.groupby('item').agg(count_sum=('frequency','sum'),percent_sum=('percent','sum')))
print('Student item totals:'); print(students.groupby('item').frequency.sum())
print('Appendix teacher N=20; student totals inferred, not assumed equal across items.')

## Validation, cleaning decisions, and export
No imputation or deduplication is applied: the source contains summaries, not respondent rows. Validation checks positive integer frequencies, category totals, and percentages. Percentages are retained unrounded in CSV; display rounding belongs in reporting.

In [ ]:
assert (teachers.frequency > 0).all() and (students.frequency > 0).all()
assert (teachers.denominator == 20).all()
assert teachers.groupby('item').frequency.sum().eq(20).all()
assert students.groupby('item').frequency.sum().eq(20).all()
assert np.allclose(teachers.groupby('item').percent.sum().values,100)
assert np.allclose(students.groupby('item').percent.sum().values,100)
teachers.to_csv(OUT/'teacher_appendix_summary.csv',index=False)
students.to_csv(OUT/'student_appendix_summary.csv',index=False)
supplementary.to_csv(OUT/'supplementary_claims_unverified.csv',index=False)
print('Wrote:',*[str(p) for p in sorted(OUT.glob('*.csv'))],sep='\n')

## Interpretation safeguards
- The table is aggregate data. Do not expand counts into pseudo-participant records or use respondent-level inferential tests.
- Appendix E Q1 (“fully/considerably responsible”, 70%) and questionnaire notes (“EFL should be separate from moral class”, 70%) appear contradictory; retain both provenance streams.
- Appendix frequencies are internally consistent at N=20. The manuscript describes a larger study; reconcile the analysis sample before publication.
- The sources include incomplete / unstructured qualitative content and no identifiable raw transcript corpus.